# Trích RTMW cho subject 1CM42_3 (85 clip)

Subject `1CM42_3` bị loại khỏi manifest gốc nên **chưa từng được trích RTMW**. Notebook này trích riêng 85 clip của subject đó bằng đúng script (`run_rtmw_splits_dual.py`), đúng checkpoint (`rtmw-dw-x-l_simcc-cocktail14_270e-384x288`) và đúng cờ mặc định (có mirror-swap) như lần trích gốc.

**An toàn:** kết quả ghi vào thư mục tạm `ipn-extract-data/keypoints_1CM42_3_staging/`, **không ghi vào `keypoints/`**. Sau khi kiểm tra vai (cell 7), mới ghép vào `keypoints/` bằng cell cuối nếu bạn muốn.

Trước khi chạy: bật **Runtime → Change runtime type → T4 GPU**, và upload `clips_1CM42_3.zip` (272 MB, trong `external_data/ipn_all_subjects/` trên máy) vào thư mục `ipn-extract-data` trên Drive.

## 1. Kiểm tra GPU và đường dẫn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Hãy bật Runtime type = T4/GPU trong Colab trước khi chạy.')
print('GPU:', torch.cuda.get_device_name(0))

from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/ipn-extract-data')
CLIPS_ZIP = DRIVE_ROOT / 'clips_1CM42_3.zip'                       # file mới upload
SCRIPT_SRC = DRIVE_ROOT / 'run_rtmw_splits_dual.py'
WEIGHTS_SRC = DRIVE_ROOT / 'rtmw-dw-x-l_simcc-cocktail14_270e-384x288-20231122.pth'
STAGING = DRIVE_ROOT / 'keypoints_1CM42_3_staging'                  # kết quả ghi vào đây trước
EXISTING_KEYPOINTS = DRIVE_ROOT / 'keypoints'                       # bộ hiện có, chỉ ĐỌC (đối chiếu)

for p in (CLIPS_ZIP, SCRIPT_SRC, WEIGHTS_SRC, EXISTING_KEYPOINTS):
    print(p, p.exists())
    if not p.exists():
        raise FileNotFoundError(f'Không tìm thấy: {p}')

## 2. Giải nén 85 clip

In [ ]:
import zipfile
WORK = Path('/content/ipn_work')
if not (WORK / 'clips').exists():
    WORK.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(CLIPS_ZIP) as z:
        z.extractall(WORK)
TRAIN_ROOT = WORK / 'clips' / 'train'
assert (TRAIN_ROOT / 'B0A' / '1CM42_3').is_dir(), 'Zip phải chứa clips/train/<lớp>/1CM42_3/*.avi'
avis = sorted(TRAIN_ROOT.rglob('*.avi'))
print('AVI:', len(avis))
assert len(avis) == 85, f'Kỳ vọng 85 clip, thấy {len(avis)}'
assert {p.parent.name for p in avis} == {'1CM42_3'}

## 3. Cài môi trường OpenMMLab (mất vài phút)

In [ ]:
# Colab mới có thể dùng Python 3.13; dùng Python 3.10 riêng cho OpenMMLab (giống notebook trích gốc).
!wget -qO /content/miniconda.sh https://repo.anaconda.com/miniconda/Miniconda3-py310_24.3.0-0-Linux-x86_64.sh
!bash /content/miniconda.sh -b -p /content/miniconda
!/content/miniconda/bin/python -m pip install -q -U 'pip<25' 'setuptools<80' wheel
!/content/miniconda/bin/python -m pip install -q 'numpy<2'
!/content/miniconda/bin/python -m pip install -q torch==2.1.2 torchvision==0.16.2 --index-url https://download.pytorch.org/whl/cu121
!/content/miniconda/bin/python -m pip install -q openmim
!/content/miniconda/bin/mim install 'mmengine==0.10.7' 'mmcv==2.1.0' 'mmdet==3.3.0'
!/content/miniconda/bin/python -m pip install -q 'chumpy' 'mmpose==1.3.1' opencv-python-headless

## 4. Chuẩn bị script và checkpoint

In [ ]:
import os, shutil
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
SCRIPT = WORK / 'run_rtmw_splits_dual.py'
WEIGHTS = WORK / WEIGHTS_SRC.name
shutil.copy2(SCRIPT_SRC, SCRIPT)
shutil.copy2(WEIGHTS_SRC, WEIGHTS)
MMPOSE_ROOT = Path('/content/miniconda/lib/python3.10/site-packages/mmpose/.mim')
OUTPUT_ROOT = WORK / 'keypoints' / 'train'      # local trước, rồi chép lên Drive staging
print('SCRIPT:', SCRIPT)
print('WEIGHTS:', WEIGHTS)
print('MMPOSE_ROOT:', MMPOSE_ROOT, MMPOSE_ROOT.exists())

## 5. Trích RTMW

In [ ]:
# 85 clip, khoảng 12.700 frame. Mặc định có mirror-swap, đúng như lần trích gốc.
!/content/miniconda/bin/python "{SCRIPT}" \
        --repo-root "{WORK}" \
        --data-root "{TRAIN_ROOT}" \
        --layout raw-subjects \
        --output-root "{OUTPUT_ROOT}" \
        --mmpose-root "{MMPOSE_ROOT}" \
        --pose-weights "{WEIGHTS}" \
        --no-cache \
        --device cuda:0

## 6. Kiểm tra đủ 85 JSON và thống kê vai

Kết quả quan trọng: `frac_one_shoulder_below_0.3` (tỉ lệ frame đúng một vai có điểm < 0,3) và `shoulder_width_px_median` so với subject đối chiếu `1CM42_4`. Gửi lại output cell này.

In [ ]:
import json, statistics as st
produced = sorted(OUTPUT_ROOT.rglob('*.json'))
print('JSON đã tạo:', len(produced), '(kỳ vọng 85)')
assert len(produced) == 85, 'Thiếu JSON: xem log cell trích để biết clip nào lỗi.'
expected = {p.stem for p in avis}
assert {p.stem for p in produced} == expected, 'Tên JSON không khớp tên clip.'
frames_expected = {p.stem: None for p in avis}

def shoulder_stats(paths, name):
    L, R, wid, nose, short = [], [], [], [], 0
    for f in paths:
        data = json.load(open(f, encoding='utf-8'))
        for fr in data:
            kp = (fr.get('instances') or [{}])[0].get('keypoints_by_name', {})
            a, b, n = kp.get('left_shoulder'), kp.get('right_shoulder'), kp.get('nose')
            if a and b:
                L.append(a['score']); R.append(b['score'])
                wid.append(abs(a['xy'][0] - b['xy'][0]))
                if n:
                    nose.append(n['xy'][0] - (a['xy'][0] + b['xy'][0]) / 2)
    if not L:
        print(name, ': không có frame nào đủ hai vai'); return None
    out = {'frames': len(L), 'score_left_median': round(st.median(L), 3), 'score_right_median': round(st.median(R), 3),
           'frac_one_shoulder_below_0.3': round(sum((l < .3) != (r < .3) for l, r in zip(L, R)) / len(L), 3),
           'frac_either_below_0.3': round(sum((l < .3) or (r < .3) for l, r in zip(L, R)) / len(L), 3),
           'shoulder_width_px_median': round(st.median(wid), 1), 'nose_minus_shoulder_mid_px_median': round(st.median(nose), 1)}
    print(name, out)
    return out

report = {'1CM42_3': shoulder_stats(produced, '1CM42_3 (mới trích)')}
ref_dir = EXISTING_KEYPOINTS / 'train' / 'G01' / '1CM42_4'
ref = sorted(ref_dir.glob('*.json')) if ref_dir.is_dir() else []
report['1CM42_4_reference'] = shoulder_stats(ref, '1CM42_4 (đối chiếu, đã có)') if ref else None
(WORK / 'shoulder_report.json').write_text(json.dumps(report, indent=2), encoding='utf-8')

## 7. Chép lên Drive (staging, không ghi đè)

In [ ]:
# Chép kết quả lên Drive vào thư mục STAGING (không ghi đè, không đụng keypoints/).
import shutil
if STAGING.exists():
    raise FileExistsError(f'Từ chối ghi đè: {STAGING} đã tồn tại. Đổi tên hoặc xóa thủ công nếu muốn làm lại.')
shutil.copytree(OUTPUT_ROOT, STAGING / 'train')
shutil.copy2(WORK / 'shoulder_report.json', STAGING / 'shoulder_report.json')
print('Đã chép', len(list((STAGING / 'train').rglob('*.json'))), 'JSON vào', STAGING)

## 8. (Tùy chọn) Ghép vào keypoints/ sau khi đã xem kết quả

In [ ]:
# CHỈ chạy cell này sau khi đã xem kết quả vai và đồng ý ghép. Không ghi đè file đã có.
MERGE = False   # đổi thành True để ghép vào keypoints/
if MERGE:
    copied = skipped = 0
    for src in sorted((STAGING / 'train').rglob('*.json')):
        dst = EXISTING_KEYPOINTS / src.relative_to(STAGING)
        if dst.exists():
            skipped += 1; continue
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst); copied += 1
    print('Đã ghép', copied, 'file; bỏ qua', skipped, 'file đã tồn tại')
else:
    print('MERGE=False: chưa ghép vào keypoints/.')